# InovaCity – Visão Computacional (Global Solution 2026.2)
**Nicho escolhido:** Infraestrutura e Pavimentação – detecção de danos no pavimento (fissuras e buracos) com bounding boxes.
**Abordagem:** transfer learning com detector pré-treinado YOLOv8n (Ultralytics) ajustado em amostra do dataset público RDD2022 (classes D00 fissura longitudinal, D10 fissura transversal, D20 fissura tipo couro de jacaré, D40 buraco).
**Importante:** o resultado da detecção apoia a triagem de inspeções e **não substitui a vistoria técnica presencial**.

Execute no Google Colab com GPU (Ambiente de execução → Alterar tipo → GPU T4). Preencha as constantes da célula 2 e rode as células em ordem. Todas as métricas do relatório devem vir das saídas reais deste notebook.

In [ ]:
!pip -q install ultralytics opencv-python-headless pandas matplotlib

In [ ]:
# ===== CONFIGURAÇÃO =====
# Baixe o RDD2022 (ver README/relatório) e descompacte em RDD_ROOT. Estrutura esperada por país:
#   RDD_ROOT/<Pais>/train/images/*.jpg   e   RDD_ROOT/<Pais>/train/annotations/xmls/*.xml  (Pascal VOC)
RDD_ROOT = '/content/RDD2022'
PAIS = 'Czech'            # escolha UM país para a amostra (ex.: Czech, Norway, Japan)
N_IMAGENS = 1500          # tamanho da amostra (reduza se faltar tempo)
SEED = 42
CLASSES = ['D00','D10','D20','D40']
WORK = '/content/data_yolo'

In [ ]:
# Baixa o RDD2022 (Czech). Se o link falhar, baixe em figshare.com (RDD2022) e envie o zip para /content
import os
os.makedirs('/content/RDD2022',exist_ok=True)
!wget -q -O /content/cz.zip https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022/RDD2022_Czech.zip
!unzip -q -o /content/cz.zip -d /content/RDD2022
!ls /content/RDD2022/*


## 1. Preparação dos dados
Passos: ler XML (Pascal VOC) → converter para YOLO → **dividir em treino/validação/teste por blocos contínuos de arquivos**, e não aleatoriamente imagem a imagem. O RDD é feito de fotos tiradas em sequência por um veículo; quadros vizinhos são quase iguais, então uma divisão aleatória colocaria "a mesma cena" em treino e teste (vazamento de dados) e inflaria as métricas. Também descartamos imagens sem anotação para a amostra.

In [ ]:
import os, glob, random, shutil, xml.etree.ElementTree as ET
from collections import Counter
random.seed(SEED)
imgs = sorted(glob.glob(f'{RDD_ROOT}/{PAIS}/train/images/*.jpg'))
def xml_de(img): return img.replace('/images/','/annotations/xmls/').replace('.jpg','.xml')
com_anot = []
for im in imgs:
    x = xml_de(im)
    if not os.path.exists(x): continue
    objs = [o for o in ET.parse(x).getroot().iter('object') if o.find('name').text in CLASSES]
    if objs: com_anot.append(im)
print(len(imgs), 'imagens;', len(com_anot), 'com anotação nas classes escolhidas')
# amostra por blocos: pega janelas contíguas para preservar a ordem das sequências
bloco = 50
blocos = [com_anot[i:i+bloco] for i in range(0, len(com_anot), bloco)]
random.shuffle(blocos)
amostra, n = [], 0
for b in blocos:
    if n >= N_IMAGENS: break
    amostra.append(b); n += len(b)
# divisão 70/15/15 por BLOCO (nunca separa quadros vizinhos entre conjuntos)
k = len(amostra); tr, va = amostra[:int(.7*k)], amostra[int(.7*k):int(.85*k)]; te = amostra[int(.85*k):]
splits = {'train': sum(tr, []), 'val': sum(va, []), 'test': sum(te, [])}
print({s: len(v) for s, v in splits.items()})

In [ ]:
def voc_para_yolo(xml_path):
    r = ET.parse(xml_path).getroot(); W = int(r.find('size/width').text); H = int(r.find('size/height').text); linhas = []
    for o in r.iter('object'):
        nome = o.find('name').text
        if nome not in CLASSES: continue
        b = o.find('bndbox'); x1,y1,x2,y2 = [float(b.find(t).text) for t in ('xmin','ymin','xmax','ymax')]
        linhas.append(f"{CLASSES.index(nome)} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
    return linhas
shutil.rmtree(WORK, ignore_errors=True)
cont = {s: Counter() for s in splits}
for s, lista in splits.items():
    os.makedirs(f'{WORK}/{s}/images'); os.makedirs(f'{WORK}/{s}/labels')
    for im in lista:
        base = os.path.basename(im)[:-4]; shutil.copy(im, f'{WORK}/{s}/images/{base}.jpg')
        L = voc_para_yolo(xml_de(im)); open(f'{WORK}/{s}/labels/{base}.txt','w').write('\n'.join(L))
        for l in L: cont[s][CLASSES[int(l.split()[0])]] += 1
open(f'{WORK}/data.yaml','w').write(f"path: {WORK}\ntrain: train/images\nval: val/images\ntest: test/images\nnames: {CLASSES}\n")
for s in cont: print(s, dict(cont[s]))

## 2. Treinamento (transfer learning)
Pesos COCO pré-treinados do YOLOv8n ajustados ao nosso conjunto. Pré-processamento: redimensionamento para 640 px (letterbox) e normalização 0–1 feitos pelo Ultralytics; aumento de dados padrão (mosaic, flip, HSV) ajuda a lidar com variação de iluminação.

In [ ]:
from ultralytics import YOLO
modelo = YOLO('yolov8n.pt')
res = modelo.train(data=f'{WORK}/data.yaml', epochs=30, imgsz=640, batch=16, seed=SEED, project='/content/runs', name='inovacity', patience=10, plots=True)

## 3. Avaliação no conjunto de TESTE (nunca visto no treino)

In [ ]:
best = YOLO('/content/runs/inovacity/weights/best.pt')
m = best.val(data=f'{WORK}/data.yaml', split='test', imgsz=640, conf=0.25, iou=0.6, plots=True, project='/content/runs', name='teste')
print('Precisão (P):', round(m.box.mp,3), ' Recall (R):', round(m.box.mr,3), ' mAP50:', round(m.box.map50,3), ' mAP50-95:', round(m.box.map,3))
import pandas as pd
pd.DataFrame({'classe':[CLASSES[i] for i in m.box.ap_class_index],'P':m.box.p,'R':m.box.r,'AP50':m.box.ap50,'AP50-95':m.box.ap}).round(3)

In [ ]:
# F1 por classe e matriz de confusão (salva em /content/runs/teste/)
from IPython.display import Image, display
for f in ('confusion_matrix.png','confusion_matrix_normalized.png','PR_curve.png','F1_curve.png'):
    p = f'/content/runs/teste/{f}'
    if os.path.exists(p): display(Image(p, width=520))

## 4. Limiar de confiança
Varremos o limiar para escolher o equilíbrio entre falsos alarmes e detecções perdidas. Em triagem de inspeção, perder um buraco real costuma ser pior que um falso alarme revisado por humano, então tende-se a um limiar moderado.

In [ ]:
linhas = []
for c in (0.10, 0.25, 0.40, 0.55):
    mm = best.val(data=f'{WORK}/data.yaml', split='test', imgsz=640, conf=c, iou=0.6, plots=False, verbose=False)
    linhas.append({'conf':c,'P':mm.box.mp,'R':mm.box.mr,'mAP50':mm.box.map50})
pd.DataFrame(linhas).round(3)

## 5. Desempenho computacional
Registre o hardware e a resolução abaixo. Latência média por imagem inclui pré-processamento, inferência e NMS (valores que o Ultralytics reporta em ms).

In [ ]:
import torch, time, platform, glob, cv2, numpy as np
hw = torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor() or platform.machine()
print('Hardware:', hw, '| torch', torch.__version__, '| resolução de inferência: 640 px')
arqs = sorted(glob.glob(f'{WORK}/test/images/*.jpg'))[:100]
best.predict(arqs[:5], imgsz=640, conf=0.25, verbose=False)           # aquecimento
t = []
for a in arqs:
    r = best.predict(a, imgsz=640, conf=0.25, verbose=False)[0]
    t.append(sum(r.speed.values()))                                   # pré + inferência + pós (ms)
t = np.array(t); print(f'Latência média: {t.mean():.1f} ms  (p95 {np.percentile(t,95):.1f} ms)  -> {1000/t.mean():.1f} FPS (imagens estáticas, uma a uma)')

## 6. Vídeo curto / sequência de quadros (monitoramento contínuo simulado)

In [ ]:
# Opção A: envie um vídeo curto (ex.: gravado de dentro de um carro, 10–20 s) para /content/video.mp4
# Opção B: se não tiver vídeo, monte a "sequência" com as imagens de teste em ordem:
VIDEO = '/content/video.mp4'
saida = '/content/saida_anotada.mp4'
if os.path.exists(VIDEO):
    cap = cv2.VideoCapture(VIDEO); fps_in = cap.get(5) or 25; W, H = int(cap.get(3)), int(cap.get(4))
    frames = iter(lambda: cap.read()[1], None)
else:
    arqs = sorted(glob.glob(f'{WORK}/test/images/*.jpg'))[:150]; fps_in = 10
    first = cv2.imread(arqs[0]); H, W = first.shape[:2]; frames = (cv2.imread(a) for a in arqs)
vw = cv2.VideoWriter(saida, cv2.VideoWriter_fourcc(*'mp4v'), fps_in, (W, H))
n = 0; t0 = time.time(); det_total = 0
for fr in frames:
    if fr is None: break
    r = best.predict(fr, imgsz=640, conf=0.25, verbose=False)[0]
    det_total += len(r.boxes); vw.write(r.plot()); n += 1
dt = time.time() - t0; vw.release()
print(f'{n} quadros em {dt:.1f}s -> {n/dt:.1f} FPS (pipeline completo incluindo leitura e escrita); {det_total} detecções')

## 7. Análise de erros
Listamos falsos positivos e falsos negativos (IoU ≥ 0,5) no teste e examinamos visualmente exemplos. Anote no relatório: sombras, poças, remendos de asfalto, marcas de pneu, faixas apagadas e iluminação forte são causas comuns de confusão.

In [ ]:
def iou(a,b):
    ix=max(0,min(a[2],b[2])-max(a[0],b[0])); iy=max(0,min(a[3],b[3])-max(a[1],b[1])); i=ix*iy
    u=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-i; return i/u if u else 0
fn, fp = [], []
for a in sorted(glob.glob(f'{WORK}/test/images/*.jpg')):
    h, w = cv2.imread(a).shape[:2]
    gt = []
    for l in open(a.replace('/images/','/labels/').replace('.jpg','.txt')).read().split('\n'):
        if l.strip():
            c,x,y,bw,bh = map(float,l.split()); gt.append((int(c),(x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h))
    r = best.predict(a, imgsz=640, conf=0.25, verbose=False)[0]
    pr = [(int(b.cls),*b.xyxy[0].tolist()) for b in r.boxes]
    for g in gt:
        if not any(p[0]==g[0] and iou(p[1:],g[1:])>=.5 for p in pr): fn.append(a)
    for p in pr:
        if not any(p[0]==g[0] and iou(p[1:],g[1:])>=.5 for g in gt): fp.append(a)
print('Falsos negativos:', len(fn), '| Falsos positivos:', len(fp))
import matplotlib.pyplot as plt
def mostrar(lista, titulo, n=6):
    fig, ax = plt.subplots(1, min(n, len(lista)), figsize=(18,4))
    ax = np.atleast_1d(ax)
    for x, a in zip(ax, lista[:n]): x.imshow(best.predict(a, conf=0.25, verbose=False)[0].plot()[:,:,::-1]); x.axis('off'); x.set_title(os.path.basename(a)[:18], fontsize=8)
    plt.suptitle(titulo); plt.show()
mostrar(list(dict.fromkeys(fn)), 'Exemplos com falso negativo (dano real não detectado)')
mostrar(list(dict.fromkeys(fp)), 'Exemplos com falso positivo (detecção sem dano anotado)')

## 8. Função de integração com o protótipo CitizenTech

In [ ]:
import json
def analisar_imagem(caminho, conf=0.25):
    """Recebe uma imagem e devolve ocorrências detectadas + confiança + aviso. Pode ser exposta via Gradio/FastAPI e chamada pelo front-end."""
    r = best.predict(caminho, imgsz=640, conf=conf, verbose=False)[0]
    dets = [{'classe':CLASSES[int(b.cls)],'confianca':round(float(b.conf),3),'caixa_xyxy':[round(v) for v in b.xyxy[0].tolist()]} for b in r.boxes]
    return {'ocorrencias':dets,'requer_vistoria_presencial':len(dets)>0,
            'aviso':'Triagem automática; não substitui a vistoria técnica presencial.'}, r.plot()[:,:,::-1]
saida_json, img_anotada = analisar_imagem(sorted(glob.glob(f'{WORK}/test/images/*.jpg'))[0])
print(json.dumps(saida_json, ensure_ascii=False, indent=2)); plt.imshow(img_anotada); plt.axis('off')

In [ ]:
# Opcional: demo em Gradio (captura de tela para o vídeo de demonstração)
!pip -q install gradio
import gradio as gr
def fn_gradio(img):
    cv2.imwrite('/tmp/in.jpg', img[:,:,::-1]); j, a = analisar_imagem('/tmp/in.jpg'); return a, json.dumps(j, ensure_ascii=False, indent=2)
gr.Interface(fn_gradio, gr.Image(type='numpy'), [gr.Image(label='Resultado'), gr.Code(label='JSON', language='json')], title='InovaCity – Danos no pavimento').launch(share=True)